# RA-NAS Analysis Notebook

This notebook visualizes:
- Accuracy vs. iteration
- Architecture diversity over time
- Agent behavior trends (activation/pooling choices)


In [ ]:
import json
from pathlib import Path
from collections import Counter
import matplotlib.pyplot as plt

exp_root = Path('../experiments')
candidates = sorted([p for p in exp_root.glob('*') if p.is_dir()])
if not candidates:
    raise RuntimeError('No experiments found under ../experiments')
latest = candidates[-1]
results = json.loads((latest / 'metrics.json').read_text())
print('Using experiment:', latest)
print('Iterations:', len(results))


In [ ]:
# 1) Accuracy vs iteration
iters = [r['iteration'] for r in results]
val_acc = [r['metrics']['val_accuracy'] for r in results]

plt.figure(figsize=(7,4))
plt.plot(iters, val_acc, marker='o', linewidth=2)
plt.xlabel('Iteration')
plt.ylabel('Validation Accuracy (%)')
plt.title('RA-NAS: Validation Accuracy Over Iterations')
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
# 2) Architecture diversity (unique architecture signatures)
def arch_signature(arch):
    return (
        arch['num_layers'],
        tuple(arch['filters']),
        tuple(arch['kernels']),
        arch['activation'],
        arch['pooling'],
        arch['use_batchnorm'],
        arch['use_dropout'],
        arch['use_skip_connections']
    )

seen = set()
diversity_curve = []
for row in results:
    seen.add(arch_signature(row['arch']))
    diversity_curve.append(len(seen))

plt.figure(figsize=(7,4))
plt.plot(iters, diversity_curve, marker='s', color='tab:green', linewidth=2)
plt.xlabel('Iteration')
plt.ylabel('Unique Architectures Seen')
plt.title('Architecture Diversity Over Iterations')
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
# 3) Agent behavior trends: activation + pooling frequencies
activation_counts = Counter([r['arch']['activation'] for r in results])
pooling_counts = Counter([r['arch']['pooling'] for r in results])

fig, axes = plt.subplots(1, 2, figsize=(10,4))
axes[0].bar(list(activation_counts.keys()), list(activation_counts.values()), color='tab:orange')
axes[0].set_title('Activation Choices')
axes[0].set_ylabel('Count')

axes[1].bar(list(pooling_counts.keys()), list(pooling_counts.values()), color='tab:blue')
axes[1].set_title('Pooling Choices')

for ax in axes:
    ax.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.show()
